# 01 — Data Inventory: `epl-25-26` (Premier League 2025-26, early-season snapshot)

What does each raw CSV in `raw_data/epl-25-26/` contain? This notebook lists every file, then goes through each one: its structure, columns, types, missing values and value ranges. It ends with a cross-file consistency check.

The data is in [FBref](https://fbref.com) format (as scraped with `soccerdata`): column names, the `match_report` / `url` paths and the multi-row header all follow FBref.

This notebook is **read-only**: it never modifies `raw_data/`.

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

# Resolve the repo root whether the notebook runs from notebooks/ or the root
ROOT = Path.cwd() if (Path.cwd() / "raw_data").exists() else Path.cwd().parent
DATA_DIR = ROOT / "raw_data" / "epl-25-26"
DATA_DIR

PosixPath('/home/srikarran/personal/premier-league/raw_data/epl-25-26')

## 1. Files at a glance

In [2]:
rows = []
for f in sorted(DATA_DIR.glob("*.csv")):
    with open(f, encoding="utf-8") as fh:
        n_lines = sum(1 for _ in fh)
    rows.append({"file": f.name, "size_kb": round(f.stat().st_size / 1024, 1), "raw_lines": n_lines})
pd.DataFrame(rows)

,file,size_kb,raw_lines
0,team_match_stats.csv,106.8,761
1,team_season_stats.csv,4.3,23


## 2. `team_match_stats.csv` — one row per team per match

Each fixture shows up **twice**, once from each team's side (`team` vs `opponent`). The file holds the **full 38-match schedule** for all 20 clubs, so unplayed fixtures are included with empty result/stat columns.

In [3]:
match = pd.read_csv(DATA_DIR / "team_match_stats.csv")
print(f"shape: {match.shape}")
match.head()

shape: (760, 23)


,league,season,team,game,date,time,round,day,venue,result,GF,GA,opponent,xG,xGA,Poss,Attendance,Captain,Formation,Opp Formation,Referee,match_report,Notes
0,ENG-Premier League,2526,Arsenal,2025-08-17 Manchester Utd-Arsenal,2025-08-17,16:30:00,Matchweek 1,Sun,Away,W,1.0,0.0,Manchester Utd,1.3,1.5,39.0,73475.0,Martin Ødegaard,4-3-3,3-4-3,Simon Hooper,/en/matches/643d26fd/Manchester-United-Arsenal...,NaN
1,ENG-Premier League,2526,Arsenal,2025-08-23 Arsenal-Leeds United,2025-08-23,17:30:00,Matchweek 2,Sat,Home,W,5.0,0.0,Leeds United,2.7,0.2,67.0,60110.0,Martin Ødegaard,4-3-3,4-3-3,Jarred Gillett,/en/matches/e8724659/Arsenal-Leeds-United-Augu...,NaN
2,ENG-Premier League,2526,Arsenal,2025-08-31 Liverpool-Arsenal,2025-08-31,16:30:00,Matchweek 3,Sun,Away,L,0.0,1.0,Liverpool,0.5,0.5,47.0,NaN,Gabriel Magalhães,4-3-3,4-2-3-1,Chris Kavanagh,/en/matches/b0005978/Liverpool-Arsenal-August-...,NaN
3,ENG-Premier League,2526,Arsenal,2025-09-13 Arsenal-Nott'ham Forest,2025-09-13,12:30:00,Matchweek 4,Sat,Home,NaN,NaN,NaN,Nott'ham Forest,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ENG-Premier League,2526,Arsenal,2025-09-21 Arsenal-Manchester City,2025-09-21,16:30:00,Matchweek 5,Sun,Home,NaN,NaN,NaN,Manchester City,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Column dictionary

| Column | Meaning |
|---|---|
| `league`, `season` | Competition (`ENG-Premier League`) and season code (`2526` = 2025-26) |
| `team` | Team this row is from the perspective of |
| `game` | Match key: `YYYY-MM-DD Home-Away` (same for both team rows of a fixture) |
| `date`, `time`, `day` | Kick-off date, local kick-off time, weekday |
| `round` | `Matchweek N` |
| `venue` | `Home` / `Away` for `team` |
| `result` | `W` / `D` / `L` for `team` (empty if not played yet) |
| `GF`, `GA` | Goals for / against |
| `opponent` | Opposing team |
| `xG`, `xGA` | Expected goals for / against |
| `Poss` | Possession % |
| `Attendance` | Crowd size |
| `Captain` | `team`'s captain in this match |
| `Formation`, `Opp Formation` | Starting formations |
| `Referee` | Match referee |
| `match_report` | Relative FBref match-report URL |
| `Notes` | Free-text notes (e.g. abandoned/awarded matches) |

### Types, missing values and cardinality

In [4]:
def profile(df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "non_null": df.notna().sum(),
        "null": df.isna().sum(),
        "null_pct": (df.isna().mean() * 100).round(1),
        "n_unique": df.nunique(),
        "example": df.apply(lambda s: s.dropna().iloc[0] if s.notna().any() else None),
    })

profile(match)

,dtype,non_null,null,null_pct,n_unique,example
league,str,760,0,0.0,1,ENG-Premier League
season,int64,760,0,0.0,1,2526
team,str,760,0,0.0,20,Arsenal
game,str,760,0,0.0,380,2025-08-17 Manchester Utd-Arsenal
date,str,760,0,0.0,55,2025-08-17
time,str,760,0,0.0,8,16:30:00
round,str,760,0,0.0,38,Matchweek 1
day,str,760,0,0.0,6,Sun
venue,str,760,0,0.0,2,Away
result,str,60,700,92.1,3,W


### Played vs. scheduled

Match-level columns are empty for fixtures that hadn't been played when the file was scraped. Here we split the rows into played and scheduled.

In [5]:
match["date"] = pd.to_datetime(match["date"])
played = match[match["result"].notna()]
scheduled = match[match["result"].isna()]

print(f"teams:            {match['team'].nunique()}")
print(f"unique fixtures:  {match['game'].nunique()}  (rows / 2 = {len(match) // 2})")
print(f"played rows:      {len(played)}  -> {played['round'].unique().tolist()}")
print(f"  date range:     {played['date'].min().date()} to {played['date'].max().date()}")
print(f"scheduled rows:   {len(scheduled)}")
print(f"  date range:     {scheduled['date'].min().date()} to {scheduled['date'].max().date()}")
played.groupby("team").size().rename("matches_played").to_frame().T

teams:            20
unique fixtures:  380  (rows / 2 = 380)
played rows:      60  -> ['Matchweek 1', 'Matchweek 2', 'Matchweek 3']
  date range:     2025-08-15 to 2025-08-31
scheduled rows:   700
  date range:     2025-09-13 to 2026-05-24


team,Arsenal,Aston Villa,Bournemouth,Brentford,Brighton,Burnley,Chelsea,Crystal Palace,Everton,Fulham,Leeds United,Liverpool,Manchester City,Manchester Utd,Newcastle Utd,Nott'ham Forest,Sunderland,Tottenham,West Ham,Wolves
matches_played,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3


### Numeric columns (played matches only)

In [6]:
played[["GF", "GA", "xG", "xGA", "Poss", "Attendance"]].describe().round(2)

,GF,GA,xG,xGA,Poss,Attendance
count,60.00,60.00,60.00,60.00,60.00,54.00
mean,1.30,1.30,1.34,1.34,50.02,41117.96
std,1.36,1.36,0.73,0.73,11.02,15607.74
min,0.00,0.00,0.20,0.20,24.00,11070.00
25%,0.00,0.00,0.70,0.70,41.75,30229.00
50%,1.00,1.00,1.20,1.20,50.00,39678.00
75%,2.00,2.00,1.72,1.72,58.25,52118.75
max,5.00,5.00,3.60,3.60,76.00,73475.00


### Categorical columns (played matches only)

In [7]:
for col in ["result", "venue", "Formation", "Referee", "day"]:
    print(f"--- {col} ---")
    print(played[col].value_counts().to_string(), "\n")

--- result ---
result
W    24
L    24
D    12 

--- venue ---
venue
Away    30
Home    30 

--- Formation ---
Formation
4-2-3-1    28
4-3-3      16
3-4-3       9
4-1-4-1     3
3-4-1-2     2
5-4-1       1
5-3-2       1 

--- Referee ---
Referee
Simon Hooper         6
Chris Kavanagh       6
Anthony Taylor       6
Peter Bankes         6
Michael Oliver       6
Jarred Gillett       4
Craig Pawson         4
Stuart Attwell       4
Samuel Barrott       4
Darren England       4
Robert Jones         4
Tony Harrington      2
Thomas Bramall       2
Michael Salisbury    2 

--- day ---
day
Sat    32
Sun    20
Fri     4
Mon     4 



### Sanity checks

- Each fixture should have exactly 2 rows.
- The two rows should mirror each other (`GF` on one = `GA` on the other; `xG` ↔ `xGA`).
- Home/away should split evenly.

In [8]:
rows_per_game = match.groupby("game").size()
print("rows per fixture:", rows_per_game.value_counts().to_dict())

pair = played.merge(
    played, left_on=["game", "team"], right_on=["game", "opponent"], suffixes=("", "_opp")
)
print("GF mirrors opp GA:  ", (pair["GF"] == pair["GA_opp"]).all())
print("xG mirrors opp xGA: ", (pair["xG"] == pair["xGA_opp"]).all())
print("venue split:        ", match["venue"].value_counts().to_dict())
print("Attendance missing on played rows:", played["Attendance"].isna().sum())

rows per fixture: {2: 380}
GF mirrors opp GA:   True
xG mirrors opp xGA:  True
venue split:         {'Away': 380, 'Home': 380}
Attendance missing on played rows: 6


## 3. `team_season_stats.csv` — one row per team, season aggregates

This is FBref's *Squad Standard Stats* table. It has a **3-row header**:
1. row 1: column group (`Playing Time`, `Performance`, `Expected`, `Progression`, `Per 90 Minutes`)
2. row 2: stat name (`MP`, `Gls`, `xG`, ...)
3. row 3: index names (`league`, `season`, `team`)

Stat names repeat across groups (e.g. `Gls` is under both `Performance` and `Per 90 Minutes`). So we read all three header rows and flatten them into unique names. Per-90 stats get a `_p90` suffix.

In [9]:
season_raw = pd.read_csv(DATA_DIR / "team_season_stats.csv", header=[0, 1, 2])
season_raw.columns.tolist()[:8]

[('Unnamed: 0_level_0', 'Unnamed: 0_level_1', 'league'),
 ('Unnamed: 1_level_0', 'Unnamed: 1_level_1', 'season'),
 ('Unnamed: 2_level_0', 'Unnamed: 2_level_1', 'team'),
 ('players_used', 'Unnamed: 3_level_1', 'Unnamed: 3_level_2'),
 ('Age', 'Unnamed: 4_level_1', 'Unnamed: 4_level_2'),
 ('Poss', 'Unnamed: 5_level_1', 'Unnamed: 5_level_2'),
 ('Playing Time', 'MP', 'Unnamed: 6_level_2'),
 ('Playing Time', 'Starts', 'Unnamed: 7_level_2')]

In [10]:
def flatten(col: tuple[str, str, str]) -> str:
    group, stat, idx = (None if str(c).startswith("Unnamed") else c for c in col)
    if idx:                          # league / season / team
        return idx
    if stat is None:                 # players_used, Age, Poss, url
        return group
    return f"{stat}_p90" if group == "Per 90 Minutes" else stat

season = season_raw.copy()
season.columns = [flatten(c) for c in season_raw.columns]
assert season.columns.is_unique
column_groups = {flatten(c): (c[0] if not c[0].startswith("Unnamed") else "") for c in season_raw.columns}

print(f"shape: {season.shape}")
season.head()

shape: (20, 35)


,league,season,team,players_used,Age,Poss,MP,Starts,Min,90s,Gls,Ast,G+A,G-PK,PK,PKatt,CrdY,CrdR,xG,npxG,xAG,npxG+xAG,PrgC,PrgP,Gls_p90,Ast_p90,G+A_p90,G-PK_p90,G+A-PK_p90,xG_p90,xAG_p90,xG+xAG_p90,npxG_p90,npxG+xAG_p90,url
0,ENG-Premier League,2526,Arsenal,22,26.2,58.3,8,88,720,8,15,10,25,13,2,2,9,0,13.6,12.1,7.6,19.6,173,361,1.87,1.25,3.12,1.62,2.87,1.71,0.95,2.65,1.51,2.46,/en/squads/18bb7c10/Arsenal-Stats
1,ENG-Premier League,2526,Aston Villa,23,28.4,55.9,8,88,720,8,8,7,15,8,0,0,13,1,6.3,6.3,4.8,11.1,119,308,1.00,0.87,1.87,1.00,1.87,0.78,0.60,1.38,0.78,1.38,/en/squads/8602292d/Aston-Villa-Stats
2,ENG-Premier League,2526,Bournemouth,23,25.8,50.5,8,88,720,8,14,10,24,13,1,1,20,0,10.4,9.6,7.0,16.6,138,293,1.75,1.25,3.00,1.62,2.87,1.30,0.88,2.18,1.20,2.08,/en/squads/4ba7cbea/Bournemouth-Stats
3,ENG-Premier League,2526,Brentford,20,25.3,41.3,8,88,720,8,11,7,18,10,1,2,15,0,10.7,9.1,7.4,16.5,103,195,1.37,0.87,2.25,1.25,2.12,1.33,0.92,2.26,1.14,2.07,/en/squads/cd051869/Brentford-Stats
4,ENG-Premier League,2526,Brighton,21,26.1,46.8,8,88,720,8,12,8,20,10,2,3,22,0,12.1,9.8,8.2,18.0,148,276,1.50,1.00,2.50,1.25,2.25,1.51,1.02,2.53,1.23,2.25,/en/squads/d07537b9/Brighton-and-Hove-Albion-S...


### Column dictionary

| Column | Group | Meaning |
|---|---|---|
| `league`, `season`, `team` | index | Competition, season code, team |
| `players_used` | — | Number of players used |
| `Age` | — | Average age, weighted by minutes played |
| `Poss` | — | Average possession % |
| `MP`, `Starts`, `Min`, `90s` | Playing Time | Matches played, starts (summed over players), minutes, minutes / 90 |
| `Gls`, `Ast`, `G+A` | Performance | Goals, assists, goals + assists |
| `G-PK`, `PK`, `PKatt` | Performance | Non-penalty goals, penalties scored, penalties attempted |
| `CrdY`, `CrdR` | Performance | Yellow / red cards |
| `xG`, `npxG` | Expected | Expected goals, non-penalty xG |
| `xAG`, `npxG+xAG` | Expected | Expected assisted goals, npxG + xAG |
| `PrgC`, `PrgP` | Progression | Progressive carries, progressive passes |
| `*_p90` | Per 90 Minutes | The matching stat divided by `90s` |
| `url` | — | Relative FBref squad URL |

In [11]:
prof = profile(season)
prof.insert(0, "group", prof.index.map(column_groups))
prof

,group,dtype,non_null,null,null_pct,n_unique,example
league,,str,20,0,0.0,1,ENG-Premier League
season,,int64,20,0,0.0,1,2526
team,,str,20,0,0.0,20,Arsenal
players_used,players_used,int64,20,0,0.0,6,22
Age,Age,float64,20,0,0.0,19,26.2
Poss,Poss,float64,20,0,0.0,19,58.3
MP,Playing Time,int64,20,0,0.0,2,8
Starts,Playing Time,int64,20,0,0.0,2,88
Min,Playing Time,int64,20,0,0.0,2,720
90s,Playing Time,int64,20,0,0.0,2,8


In [12]:
season.describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
season,20.0,2526.00,0.00,2526.00,2526.00,2526.00,2526.00,2526.00
players_used,20.0,22.00,1.78,19.00,20.75,22.00,23.00,25.00
Age,20.0,26.67,1.13,24.40,25.75,26.75,27.38,28.50
Poss,20.0,50.05,6.92,35.10,46.12,49.45,55.45,61.90
MP,20.0,8.10,0.31,8.00,8.00,8.00,8.00,9.00
Starts,20.0,89.10,3.39,88.00,88.00,88.00,88.00,99.00
Min,20.0,729.00,27.70,720.00,720.00,720.00,720.00,810.00
90s,20.0,8.10,0.31,8.00,8.00,8.00,8.00,9.00
Gls,20.0,10.15,3.57,4.00,7.75,9.00,13.25,16.00
Ast,20.0,7.15,2.92,3.00,5.00,7.00,10.00,13.00


## 4. Cross-file consistency

- Do both files use the same team names?
- Do they cover the same point in the season?

In [13]:
match_teams, season_teams = set(match["team"]), set(season["team"])
print("team names identical:", match_teams == season_teams)
print("only in match file:  ", match_teams - season_teams or "-")
print("only in season file: ", season_teams - match_teams or "-")

coverage = (
    season.set_index("team")[["MP"]]
    .join(played.groupby("team").size().rename("played_in_match_file"))
    .assign(gap=lambda d: d["MP"] - d["played_in_match_file"])
)
coverage

team names identical: True
only in match file:   -
only in season file:  -


,MP,played_in_match_file,gap
team,,,
Arsenal,8,3,5
Aston Villa,8,3,5
Bournemouth,8,3,5
Brentford,8,3,5
Brighton,8,3,5
Burnley,8,3,5
Chelsea,8,3,5
Crystal Palace,8,3,5
Everton,8,3,5


## 5. Summary

| File | Grain | Rows | Notes |
|---|---|---|---|
| `team_match_stats.csv` | team × match | 760 (20 teams × 38) | Full fixture list. Only **Matchweeks 1–3** (60 rows, Aug 15–31 2025) have results/stats; the other 700 rows are unplayed fixtures. |
| `team_season_stats.csv` | team | 20 | FBref squad standard stats with a 3-row header. Covers **8–9 matches per team** (Leeds and West Ham have 9). |

**Things to keep in mind downstream**
- **The two files are out of sync.** The match file stops at MW3; the season aggregates run to about MW8–9. Season totals can't be rebuilt from the match file as it stands. Both files are partial snapshots from about Oct 2025, not the full season.
- **Match results for the full season are now in `raw_data/2526.csv`** (see notebook 06). Clean data takes 2025-26 results from there; the 30 played matches in this folder agree with it on every score and date. The FBref-only stats here (xG, possession, formations, …) still cover MW1–3 only.
- Unplayed fixtures should be filtered on `result.notna()` before any match-level analysis.
- `Attendance` is missing on some played rows even though the result is there.
- `Notes` is empty throughout. It's read as `float64` because it's all-NaN.
- The season file needs `header=[0, 1, 2]` plus flattening; see `flatten()` above.